# IBKR现金利息来源核验
保存47个月官方请求与空响应证据，严格解析百分数及前日标记；只将真实观测写入Qlib，不回填空月份，不更改原回测收益。

In [1]:
from pathlib import Path
import os,json,hashlib
import pandas as pd,numpy as np,qlib
from qlib.workflow import R
from src.data.ibkr_interest_source import parse_benchmark_month
from src.data.qlib_native import build_qlib_store_from_frame,load_qlib_feature_frame,QlibProviderConfig
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/ibkr_credit_interest_source_20261015';p=json.loads((card/'plan.json').read_text());manifest_path=root/p['source_manifest'];assert hashlib.sha256(manifest_path.read_bytes()).hexdigest()==p['source_manifest_sha256'];m=json.loads(manifest_path.read_text());raw=manifest_path.parent
parts=[];receipts=[];payloads={}
for e in m['entries']:
 path=raw/e['path'];data=path.read_bytes();assert hashlib.sha256(data).hexdigest()==e['sha256'];payloads[e['path']]=data
 frame=parse_benchmark_month(data.decode(),month=e['month'],as_of=p['as_of']);assert len(frame)==e['observed_rows'];parts.append(frame);receipts.append({'month':e['month'],'observed_rows':len(frame),'known_usd_rows':int(frame.benchmark_rate.notna().sum()),'status':'observed' if len(frame) else 'header_only_no_data','url':e['url'],'sha256':e['sha256']})
for e in m['rule_pages']:
 data=(raw/e['path']).read_bytes();assert hashlib.sha256(data).hexdigest()==e['sha256'];payloads[e['path']]=data
frame=pd.concat(parts,ignore_index=True).sort_values('datetime');assert not frame.datetime.duplicated().any();assert frame.benchmark_rate.notna().all();frame['instrument']='USD_BENCHMARK';provider=root/'data/qlib/ibkr_credit_interest_20261015_v1'
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';manifest=build_qlib_store_from_frame(frame,provider_uri=provider,as_of=p['as_of'],source='Official IBKR LLC observed monthly USD benchmark downloads',field_map={'benchmark_rate':'benchmark_rate','prior_day_flag':'prior_day_flag'})
qlib.init(provider_uri=str(provider),region='us');quotes=load_qlib_feature_frame(QlibProviderConfig(provider),instruments=['USD_BENCHMARK'],fields=['$benchmark_rate','$prior_day_flag'],start=str(frame.datetime.min().date()),end=p['as_of']);np.testing.assert_allclose(quotes['$benchmark_rate'],frame.benchmark_rate.to_numpy(),rtol=0,atol=1e-8)
policy={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/raw_payloads','inputs/source_manifest','analysis/quotes']}
with R.start(experiment_name=card.name,uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=p,raw_payloads=payloads,source_manifest=m,parser=(root/'src/data/ibkr_interest_source.py').read_text());rec.save_objects(artifact_path='analysis',quotes=quotes,provider_manifest=manifest,coverage=pd.DataFrame(receipts));rec.log_metrics(requested_months=len(receipts),observed_rows=len(frame),registered_artifact_count=7)
summary={'experiment':card.name,'decision':'needs_data_historical_benchmarks_and_settlement_ledger','new_fits':0,'new_backtests':0,'requested_months':len(receipts),'header_only_months':sum(e['status']=='header_only_no_data' for e in receipts),'observed_usd_rows':len(frame),'first_observed_date':str(frame.datetime.min().date()),'last_observed_date':str(frame.datetime.max().date()),'historical_backtest_coverage_complete':False,'historical_credit_spread_contract_verified':False,'settled_cash_ledger_implemented':False,'cash_interest_added_to_existing_returns':False,'forecast_input_qualified':False,'provider_uri':str(provider),'region':'us','as_of':p['as_of'],'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'recorder_local_dir':str(card/'qlib_recorder'/str(rec.experiment_id)/rec.id),'artifact_policy':policy,'objective_achieved':False,'scope':p['scope']}
pd.DataFrame(receipts).to_csv(card/'monthly_coverage.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'artifact_policy':policy}))
binding=json.loads((root/'research/opening_state_training_20261015/nas_binding.json').read_text());binding.pop('evidence',None);binding.update(source_id=p['source_manifest_sha256'],raw_authority='IBKR official observed monthly benchmarks and rule page snapshots');(card/'nas_binding.json').write_text(json.dumps(binding,indent=2))
a=Path(summary['recorder_local_dir'])/'artifacts';(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(f.relative_to(card)),'sha256':hashlib.sha256(f.read_bytes()).hexdigest()} for f in sorted(a.rglob('*')) if f.is_file()]},indent=2))
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));(card/'summary.json').write_text(json.dumps(summary,indent=2));print({k:v for k,v in summary.items() if k in ['observed_usd_rows','first_observed_date','last_observed_date','header_only_months','decision']})


# IBKR现金利息：发现账务缺项，但历史来源尚不足

现有SOXL/SOXX/CASH研究沿用现金零利息，可能低估现金收益；本轮未向任何账户添加利息，也未更改已报告收益。完成官方来源获取、解析、Qlib入库读回及NAS证据归档，0拟合、0新增回测。

核查官方LLC月度基准下载端点，共请求2022-12至2026-10的47个月。38个月HTTP成功但仅返回表头，不能当0利率。2026-02至2026-10有182条USD观测，最早2026-02-02、最晚2026-10-13；缺少完整2023–2025和2026年1月数据。本结论只描述本次端点响应，不断言所有官方渠道均无更早历史。月度逐项URL、哈希和行数见monthly_coverage.csv；原始空响应及非空响应全部在Recorder保留。

官方规则入口：
- [现金利率与分层](https://www.interactivebrokers.com/en/accounts/fees/pricing-interest-rates.php)：美元首10,000现金免息，NAV门槛及Pro利差影响付息。
- [计息计算](https://www.interactivebrokers.com/en/pricing/pricing-calculations-int.php)：使用已结算现金；NAV低于100,000时按比例缩减；一般USD采用360天基数；应计与最终月度入账分开。
- [基准定义](https://www.interactivebrokers.com/en/pricing/reference-benchmark-rates-int.php)：IBKR基准由多种市场参考综合确定，不能直接用Fed Funds或当前利率代替历史。
- [历史月度入口](https://www.interactivebrokers.com/en/accounts/fees/monthlyInterestRates.php)：星号表示沿用前日数据，本轮单独保留标记，不假定已获当日新定值。

这些是当前页面规则快照，不是2023以来每期合同和点时版本认证。源数据as_of为2026-10-14，与旧价格回测截至2026-10-09分开；之后的观测不会进入旧回测。基准百分数字段转换为小数（3.880%→0.0388），N/A保留未知，空月份不填充，负值括号和前日标记均测试；请求月份错位及超过as_of会拒绝。仅真实观测进入Qlib，不做跨缺口前向填充。

现有账簿ending_cash是交易后即时现金，不等于官方计息的已结算现金。完整复现还需要交易结算日历/结算延迟、日终NAV、应计余额与月度支付处理，并同时重算SOXX基准；利息改变净值和风险路径，不能只加到策略期末财富。应计余额亦不能简单当作立即复利的已结算现金。现阶段不声称已完成利息引擎或精确IBKR账单复现。

决策needs_data_historical_benchmarks_and_settlement_ledger：历史基准覆盖、历史利差合同及结算账本均未合格。已取得来源可复用，但不是模型新因子，也不证明现金利息能填平超额缺口。后续先核查其他可验证官方历史材料或既有账单证据，再决定完整结算实现；不得用近月利率回填历史来获得漂亮回测。没有请求账户凭证或发送外部消息，生产未改，稳定净SOXX超额目标未完成。

1项针对性解析合同测试通过；执行notebook及原生Qlib读回通过；NAS冷恢复逐对象哈希通过。源HTML和全币种下载为Recorder证据，不放大卡根目录。
